# Stream A — Pyannote Speaker Diarization

Standalone notebook for speaker diarization using Pyannote.
Run this in the `streamA_diar` conda environment.

**Output:** Per-subject JSON files in `outputs/cache/diarization/`
These are read by `streamA_full_scale.ipynb` automatically.

```
conda create -n streamA_diar python=3.10
conda activate streamA_diar
pip install torch==2.2.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu121
pip install pyannote.audio==3.3.2
pip install tqdm
```

---
## Cell 0 — Configuration

In [ ]:
import os, re, json, logging
from pathlib import Path
from datetime import datetime
import torch
from tqdm.auto import tqdm

# ── Edit this section ────────────────────────────────────────────────
RAW_DATA_DIR      = Path("/home3/tmp/u/lily/whisperx")
DIARIZATION_CACHE = Path("/home3/tmp/u/lily/whisperx/outputs/cache/diarization")
LOG_DIR           = Path("/home3/tmp/u/lily/whisperx/outputs/logs")

HF_TOKEN          = "your_token_here"  # clear before git push
PYANNOTE_MODEL    = "pyannote/speaker-diarization-3.1"

N_SUBJECTS        = 10    # set to None to run all 189
EXCLUDED_SUBJECTS = {"342", "394", "398", "460"}
MIN_SEGMENT_DUR_SEC = 0.5
# ─────────────────────────────────────────────────────────────────────

DIARIZATION_CACHE.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Logger
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = LOG_DIR / f"diarization_{RUN_ID}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(log_path), logging.StreamHandler()],
)
logger = logging.getLogger("diarization")

print(f"Data dir  : {RAW_DATA_DIR}")
print(f"Cache dir : {DIARIZATION_CACHE}")
print(f"Device    : {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")

---
## Cell 1 — Subject Discovery

In [ ]:
def discover_subjects(raw_dir: Path, excluded: set) -> list:
    subjects = []
    audio_files = sorted(
        raw_dir.glob("*_AUDIO.wav"),
        key=lambda p: int(re.search(r"(\d+)_AUDIO", p.name).group(1))
    )
    for audio in audio_files:
        m = re.match(r"(\d+)_AUDIO\.wav", audio.name)
        if not m:
            continue
        session_id = m.group(1)
        subject_id = f"{session_id}_P"
        if session_id in excluded:
            continue
        subjects.append({
            "subject_id": subject_id,
            "session_id": session_id,
            "audio_path": audio,
        })
    return subjects

SUBJECTS = discover_subjects(RAW_DATA_DIR, EXCLUDED_SUBJECTS)

if N_SUBJECTS:
    SUBJECTS = SUBJECTS[:N_SUBJECTS]
    print(f"DRY RUN: using first {N_SUBJECTS} subjects")

print(f"Subjects to diarize: {len(SUBJECTS)}")
print(f"Sample: {[s['subject_id'] for s in SUBJECTS[:5]]}")

---
## Cell 2 — Load Pyannote

In [ ]:
from pyannote.audio import Pipeline as PyannotePipeline

# ── Check cache first — skip model load if all done ───────────────────
needs_diarization = [
    subj for subj in SUBJECTS
    if not (DIARIZATION_CACHE / f"{subj['subject_id']}.json").exists()
]
print(f"Total subjects  : {len(SUBJECTS)}")
print(f"From cache      : {len(SUBJECTS) - len(needs_diarization)}")
print(f"Need processing : {len(needs_diarization)}")

if needs_diarization:
    if not HF_TOKEN or HF_TOKEN == "your_token_here":
        raise ValueError("Set HF_TOKEN in Cell 0 before running.")
    logger.info(f"Loading Pyannote: {PYANNOTE_MODEL}")
    pyannote_pipeline = PyannotePipeline.from_pretrained(
        PYANNOTE_MODEL, use_auth_token=HF_TOKEN
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pyannote_pipeline.to(device)
    print(f"Pyannote loaded on: {device}")
else:
    pyannote_pipeline = None
    print("All cached — skipping model load ✅")

---
## Cell 3 — Run Diarization

In [ ]:
def diarize_audio(pipeline, audio_path: Path) -> list:
    diarization = pipeline(str(audio_path))
    return [
        {"start": turn.start, "end": turn.end, "speaker": speaker}
        for turn, _, speaker in diarization.itertracks(yield_label=True)
    ]

def identify_participant(diar_segments: list) -> str:
    """Speaker with most total speaking time = Participant."""
    durations = {}
    for seg in diar_segments:
        spk = seg["speaker"]
        durations[spk] = durations.get(spk, 0) + (seg["end"] - seg["start"])
    return max(durations, key=durations.get) if durations else None

def filter_participant(diar_segments: list, participant_label: str,
                        min_dur: float = MIN_SEGMENT_DUR_SEC) -> list:
    return [
        seg for seg in diar_segments
        if seg["speaker"] == participant_label
        and (seg["end"] - seg["start"]) >= min_dur
    ]


ALL_DIAR_SEGMENTS = {}

for subj in tqdm(SUBJECTS, desc="Diarization"):
    sid        = subj["subject_id"]
    cache_path = DIARIZATION_CACHE / f"{sid}.json"

    if cache_path.exists():
        with open(cache_path) as f:
            ALL_DIAR_SEGMENTS[sid] = json.load(f)
        continue

    try:
        print(f"Processing {sid}...")
        diar_segs         = diarize_audio(pyannote_pipeline, subj["audio_path"])
        participant_label = identify_participant(diar_segs)
        participant_segs  = filter_participant(diar_segs, participant_label)

        with open(cache_path, "w") as f:
            json.dump(participant_segs, f)

        ALL_DIAR_SEGMENTS[sid] = participant_segs
        logger.info(f"{sid}: {len(participant_segs)} participant segments")

    except Exception as e:
        logger.error(f"{sid}: diarization failed - {e}")
        ALL_DIAR_SEGMENTS[sid] = []

total_segs = sum(len(v) for v in ALL_DIAR_SEGMENTS.values())
print(f"\nDiarization complete")
print(f"  Subjects processed         : {len(ALL_DIAR_SEGMENTS)}")
print(f"  Total participant segments : {total_segs}")
print(f"  Avg per subject            : {total_segs / max(len(SUBJECTS),1):.0f}")
print(f"\nResults saved to: {DIARIZATION_CACHE}")
print("streamA_full_scale.ipynb will read these automatically ✅")